<a href="https://colab.research.google.com/github/Aivon99/BigDataAndTextMiningProject/blob/Ivo/tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Data Analytics and Text Mining Project

A standalone notebook of quick checks/unit tests for the recent additions -- meant to be run on Colab and reported back, not part of the main pipeline. Each section is self-contained and prints a clear PASS/FAIL-style result; the final cell prints a consolidated summary.

Covers:
1. Model checkpoint + processor load
2. Vision-token geometry (patch_size / merge_size / min_pixels / max_pixels)
3. Empirical patch-alignment check (via a real processed image's `image_grid_thw`, not just arithmetic)
4. Loading the real generated datasets from Hugging Face (should be ~12k samples total across the 3 tasks)
5. Patch-reordering primitives (`get_patch_reordering_indices` / `apply_patch_permutation`) -- pure CPU, no model needed
6. `PlackettLucePatchPolicy` (Learned Reordering) sanity check -- pure CPU, no model needed
7. `find_resumable_checkpoint` against a repo that doesn't exist -- should return `None` cleanly, not crash
8. `config.yaml` loading + `clamp_balanced_turn_counts` logic

In [1]:
!rm -rf /content/BigDataAndTextMiningProject


In [2]:
!pip install -q --upgrade "pillow<11.0.0" transformers>=4.45.0 accelerate torchao scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub python-dotenv PyYAML

# Standard library imports
import os
import subprocess
import sys
from pathlib import Path

# Third-party library imports
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import login, HfApi
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor


## Setup

In [3]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "Ivo",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    if repo_root.exists():
        print(f"Repository directory already exists at: {repo_root}")
    else:
        repo_url = f"https://github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"
        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve()

print(f"Setup Complete. REPO_ROOT: {repo_root}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))


Cloning repository from https://github.com/Aivon99/BigDataAndTextMiningProject.git...
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [4]:
from data.generation import render_board_svg
from data.utilities import load_config, clamp_balanced_turn_counts
from eval.utilities import (
    get_patch_reordering_indices,
    apply_patch_permutation,
    reorder_chessboard_image,
    find_resumable_checkpoint,
)
from model.patch_geometry import get_vision_token_pitch, check_alignment
from training.learned_reordering import PlackettLucePatchPolicy

import chess

print("All custom modules imported successfully!")

# Results collected across all tests for the final summary cell.
results = {}


All custom modules imported successfully!


## Authenticate with Hugging Face

Adjust `ORGANIZATION_NAME` below to wherever the real ~12k-sample dataset was actually pushed (defaults to `bdatm-project`, since that's what `Datasets_creator.ipynb`'s upload cell is currently set to -- change it if that's not where the real run went).

In [5]:
ORGANIZATION_NAME = "bdatm-project"  # <-- change if you pushed the real dataset elsewhere
MODEL_ID = "Qwen/Qwen3.5-0.8B"

hf_token = None
if CONFIG["colab"]:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    from dotenv import load_dotenv
    load_dotenv(repo_root / ".env")
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub using HF_TOKEN.")
else:
    print("No HF_TOKEN found in secrets/.env/environment -- falling back to interactive login.")
    login()

Logged in to Hugging Face Hub using HF_TOKEN.


## Test 1: Model checkpoint & processor load

Confirms `MODEL_ID` actually resolves and loads (both the processor, which the geometry tests below need, and the full model, needed by everything else in the pipeline).

In [6]:
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    print(f"[Test 1a] PASS -- processor loaded for '{MODEL_ID}'")
    processor_ok = True
except Exception as e:
    print(f"[Test 1a] FAIL -- could not load processor for '{MODEL_ID}': {e}")
    processor_ok = False

try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    print(f"[Test 1b] PASS -- model loaded for '{MODEL_ID}'")
    model_ok = True
except Exception as e:
    print(f"[Test 1b] FAIL -- could not load model for '{MODEL_ID}': {e}")
    model = None
    model_ok = False

results["1_model_and_processor_load"] = processor_ok and model_ok


HTTP Error 429 thrown while requesting HEAD https://huggingface.co/Qwen/Qwen3.5-0.8B/resolve/main/processor_config.json
Rate limited. Waiting 43.0s before retry [Retry 1/5].
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


[Test 1a] PASS -- processor loaded for 'Qwen/Qwen3.5-0.8B'


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[Test 1b] PASS -- model loaded for 'Qwen/Qwen3.5-0.8B'


## Test 2: Vision-token geometry

Reads `patch_size` / `merge_size` / `min_pixels` / `max_pixels` directly off the loaded processor -- this is the ground truth for Patch Resolution Alignment, not the unverified family defaults `patch_geometry.py` falls back to when no processor is given.

In [7]:
if processor_ok:
    image_processor = getattr(processor, "image_processor", None)
    patch_size = getattr(image_processor, "patch_size", None)
    merge_size = getattr(image_processor, "merge_size", None)
    min_pixels = getattr(image_processor, "min_pixels", None)
    max_pixels = getattr(image_processor, "max_pixels", None)

    print(f"patch_size:  {patch_size}")
    print(f"merge_size:  {merge_size}")
    print(f"min_pixels:  {min_pixels}")
    print(f"max_pixels:  {max_pixels}")

    pitch = get_vision_token_pitch(processor)
    print(f"\nget_vision_token_pitch(processor) -> {pitch}px per vision token")

    results["2_vision_token_geometry"] = patch_size is not None and merge_size is not None
    if results["2_vision_token_geometry"]:
        print("[Test 2] PASS -- patch_size/merge_size read from the real processor")
    else:
        print("[Test 2] FAIL -- processor did not expose patch_size/merge_size (see patch_geometry.py fallback warning above)")
else:
    print("[Test 2] SKIPPED -- Test 1 (processor load) did not pass")
    results["2_vision_token_geometry"] = False


patch_size:  16
merge_size:  2
min_pixels:  None
max_pixels:  None

get_vision_token_pitch(processor) -> 32px per vision token
[Test 2] PASS -- patch_size/merge_size read from the real processor


## Test 3: Empirical patch-alignment check

Renders one real board image, resizes it to a few candidate resolutions (including `512`, the size actually used for the already-generated 12k-sample dataset), runs each through the **real processor**, and reads `image_grid_thw` off the result -- this is what the model actually receives, not an arithmetic prediction. Flags whether each candidate divides cleanly into the 8x8 chess grid.

In [8]:
if processor_ok:
    board = chess.Board()  # standard starting position is enough for a geometry check
    base_image = render_board_svg(board=board, size=512)

    candidate_sizes = [512, 224, 448, 336, 672]

    print(f"{'image_size':>10} | {'grid_h':>6} {'grid_w':>6} | {'tokens/side':>11} | {'tokens/square':>13} | aligned?")
    print("-" * 70)

    alignment_report = {}
    for size in candidate_sizes:
        test_image = base_image.resize((size, size))
        inputs = processor(text=["<|image_pad|>"], images=[test_image], return_tensors="pt")

        grid_thw = inputs.get("image_grid_thw")
        if grid_thw is None:
            print(f"{size:>10} | image_grid_thw not present in processor output -- can't verify")
            continue

        # image_grid_thw is [t, h, w] in patch units (post spatial-merge)
        _, grid_h, grid_w = grid_thw[0].tolist()
        tokens_per_side = grid_h  # square renders -> grid_h == grid_w expected
        tokens_per_square = tokens_per_side / 8
        is_aligned = grid_h == grid_w and tokens_per_square.is_integer()
        alignment_report[size] = is_aligned

        print(f"{size:>10} | {grid_h:>6} {grid_w:>6} | {tokens_per_side:>11} | {tokens_per_square:>13.3f} | {'YES' if is_aligned else 'no'}")

    results["3_empirical_alignment"] = any(alignment_report.values())
    print()
    if alignment_report.get(512):
        print("[Test 3] Current dataset resolution (512) IS aligned -- no regeneration needed.")
    else:
        aligned_candidates = [s for s, ok in alignment_report.items() if ok]
        print(
            f"[Test 3] Current dataset resolution (512) is NOT aligned. "
            f"Aligned candidates found: {aligned_candidates or 'none in this candidate list -- try more sizes'}"
        )
else:
    print("[Test 3] SKIPPED -- Test 1 (processor load) did not pass")
    results["3_empirical_alignment"] = False


image_size | grid_h grid_w | tokens/side | tokens/square | aligned?
----------------------------------------------------------------------
       512 |     32     32 |          32 |         4.000 | YES
       224 |     16     16 |          16 |         2.000 | YES
       448 |     28     28 |          28 |         3.500 | no
       336 |     20     20 |          20 |         2.500 | no
       672 |     42     42 |          42 |         5.250 | no

[Test 3] Current dataset resolution (512) IS aligned -- no regeneration needed.


## Test 4: Load the real generated datasets from Hugging Face

Confirms the ~12k-sample dataset (3 tasks x ~4000 puzzles = ~12k images) is reachable and looks as expected: right split sizes, right schema, and confirms what resolution the images were actually saved at.

In [9]:
#Removed

## Test 5: Patch-reordering primitives (no GPU needed)

Sanity-checks `get_patch_reordering_indices` (every strategy must be a valid permutation of 0..63) and `apply_patch_permutation` (output image must keep the same size; raster order must be a no-op).

In [10]:
test5_ok = True

for strategy in ["raster", "zigzag", "spiral", "file_wise", "rank_wise"]:
    indices = get_patch_reordering_indices(strategy=strategy, grid_size=8)
    is_valid_permutation = sorted(indices) == list(range(64))
    print(f"[{strategy:>10}] valid permutation of 0..63: {is_valid_permutation}")
    test5_ok = test5_ok and is_valid_permutation

board = chess.Board()
test_image = render_board_svg(board=board, size=512)

raster_indices = get_patch_reordering_indices(strategy="raster", grid_size=8)
raster_result = apply_patch_permutation(test_image, raster_indices, grid_size=8, img_size=512)
raster_is_noop = list(raster_result.getdata()) == list(test_image.resize((512, 512)).getdata())
print(f"raster reorder is a pixel-exact no-op: {raster_is_noop}")
test5_ok = test5_ok and raster_is_noop

zigzag_result = reorder_chessboard_image(test_image, strategy="zigzag", grid_size=8)
size_preserved = zigzag_result.size == (512, 512)
print(f"reordered image size preserved: {size_preserved}")
test5_ok = test5_ok and size_preserved

results["5_reordering_primitives"] = test5_ok
print(f"[Test 5] {'PASS' if test5_ok else 'FAIL'}")


[    raster] valid permutation of 0..63: True
[    zigzag] valid permutation of 0..63: True
[    spiral] valid permutation of 0..63: True
[ file_wise] valid permutation of 0..63: True
[ rank_wise] valid permutation of 0..63: True
raster reorder is a pixel-exact no-op: True
reordered image size preserved: True
[Test 5] PASS


## Test 6: `PlackettLucePatchPolicy` sanity check (no GPU needed)

Confirms the Learned Reordering policy module actually runs: sampling produces valid permutations with the right shapes, and a REINFORCE backward pass successfully flows gradients into `policy.logits`.

In [11]:
policy = PlackettLucePatchPolicy(grid_size=8)

batch_size = 4
permutation, log_prob = policy.sample(batch_size=batch_size)

shape_ok = permutation.shape == (batch_size, 64) and log_prob.shape == (batch_size,)
print(f"permutation shape {tuple(permutation.shape)}, log_prob shape {tuple(log_prob.shape)}: {'OK' if shape_ok else 'WRONG'}")

all_valid_perms = all(sorted(permutation[i].tolist()) == list(range(64)) for i in range(batch_size))
print(f"every sampled row is a valid permutation of 0..63: {all_valid_perms}")

dummy_reward = torch.randn(batch_size)
policy.update_baseline(dummy_reward)
loss = policy.reinforce_loss(dummy_reward, log_prob)

grad_ok = False
try:
    loss.backward()
    grad_ok = policy.logits.grad is not None and torch.isfinite(policy.logits.grad).all()
    print(f"backward() succeeded, policy.logits.grad finite: {grad_ok}")
except Exception as e:
    print(f"backward() FAILED: {e}")

test6_ok = shape_ok and all_valid_perms and grad_ok
results["6_learned_reordering_policy"] = test6_ok
print(f"[Test 6] {'PASS' if test6_ok else 'FAIL'}")


permutation shape (4, 64), log_prob shape (4,): OK
every sampled row is a valid permutation of 0..63: True
backward() succeeded, policy.logits.grad finite: True
[Test 6] PASS


## Test 7: `find_resumable_checkpoint` against a nonexistent repo

Should return `None` cleanly (no exception) when the target repo doesn't exist yet -- this is the "starting fresh, nothing to resume" path every fine-tuning cell hits on a first run.

In [12]:
nonexistent_repo = f"{ORGANIZATION_NAME}/this-repo-should-not-exist-test-checkpoint"

try:
    result = find_resumable_checkpoint(nonexistent_repo)
    test7_ok = result is None
    print(f"find_resumable_checkpoint('{nonexistent_repo}') -> {result!r}")
except Exception as e:
    test7_ok = False
    print(f"find_resumable_checkpoint raised an exception instead of returning None: {e}")

results["7_resumable_checkpoint_missing_repo"] = test7_ok
print(f"[Test 7] {'PASS' if test7_ok else 'FAIL'}")


find_resumable_checkpoint('bdatm-project/this-repo-should-not-exist-test-checkpoint') -> None
[Test 7] PASS


## Test 8: `config.yaml` loading + `clamp_balanced_turn_counts`

Confirms the config file parses with the expected keys, and that the clamping logic actually reduces an over-large request instead of crashing.

In [13]:
test8_ok = True

try:
    cfg = load_config(repo_root / "configs" / "config.yaml")["dataset_generation"]
    expected_keys = {"csv_path", "random_seed", "max_samples", "n_per_turn", "image_size"}
    keys_ok = expected_keys.issubset(cfg.keys())
    print(f"config.yaml loaded: {cfg}")
    print(f"has all expected keys: {keys_ok}")
    test8_ok = test8_ok and keys_ok
except Exception as e:
    print(f"FAIL -- could not load config.yaml: {e}")
    test8_ok = False

# Synthetic small dataframe: only 3 white-to-move, 5 black-to-move rows available.
# Requesting n_per_turn=100 should clamp down to 3 (the scarcer side), not crash.
synthetic_df = pd.DataFrame({"turn": ["w"] * 3 + ["b"] * 5})
clamped = clamp_balanced_turn_counts(synthetic_df, n_per_turn=100)
clamp_ok = clamped == 3
print(f"clamp_balanced_turn_counts(3 white / 5 black, requested=100) -> {clamped} (expected 3): {clamp_ok}")
test8_ok = test8_ok and clamp_ok

results["8_config_and_clamping"] = test8_ok
print(f"[Test 8] {'PASS' if test8_ok else 'FAIL'}")


config.yaml loaded: {'csv_path': 'lichess_db_puzzle.csv', 'random_seed': 42, 'max_samples': 20000, 'n_per_turn': 2000, 'image_size': 512}
has all expected keys: True
[clamp_balanced_turn_counts] Requested n_per_turn=100 exceeds what's available (white=3, black=5); clamping to 3.
clamp_balanced_turn_counts(3 white / 5 black, requested=100) -> 3 (expected 3): True
[Test 8] PASS


## Summary

In [14]:
print("=== TEST SUMMARY ===")
for name, passed in results.items():
    print(f"{'PASS' if passed else 'FAIL'} -- {name}")

all_passed = all(results.values())
print(f"\n{'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED -- see details above'}")


=== TEST SUMMARY ===
PASS -- 1_model_and_processor_load
PASS -- 2_vision_token_geometry
PASS -- 3_empirical_alignment
PASS -- 5_reordering_primitives
PASS -- 6_learned_reordering_policy
PASS -- 7_resumable_checkpoint_missing_repo
PASS -- 8_config_and_clamping

ALL TESTS PASSED
